# YOLO11n 교통약자 감지 파인튜닝
**대상 클래스**: Wheelchair, Crutch  
**데이터셋**: Open Images V7 (train 961장 + val 50장)  
**목표**: 라즈베리파이 5 CPU 추론용 경량 모델

## 사전 준비
1. 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
2. Google Drive에 `sbms-pi/mobility_aids.zip` 업로드 완료 확인

In [ ]:
# 1. Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. 데이터셋 압축 해제
import zipfile, os

zip_path = '/content/drive/MyDrive/sbms-pi/mobility_aids.zip'
extract_path = '/content/datasets'

os.makedirs(extract_path, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

print('압축 해제 완료')
!ls /content/datasets/mobility_aids/

In [ ]:
# 3. ultralytics 설치
!pip install ultralytics -q

In [ ]:
# 4. dataset.yaml 경로 수정 (Colab 경로로)
import yaml

yaml_path = '/content/datasets/mobility_aids/dataset.yaml'
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)

cfg['path'] = '/content/datasets/mobility_aids'

with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f)

print('dataset.yaml 수정 완료:')
!cat /content/datasets/mobility_aids/dataset.yaml

In [ ]:
# 5. YOLO11n 파인튜닝 (T4 GPU, ~1시간)
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

results = model.train(
    data='/content/datasets/mobility_aids/dataset.yaml',
    epochs=100,
    batch=32,
    imgsz=640,
    device=0,          # T4 GPU
    patience=20,       # early stopping
    workers=2,
    project='/content/runs',
    name='mobility_yolo11n',
    exist_ok=True,
)

print('학습 완료!')
print('Best model:', results.save_dir)

In [ ]:
# 6. 학습 결과 확인
import glob

best = '/content/runs/mobility_yolo11n/weights/best.pt'
last = '/content/runs/mobility_yolo11n/weights/last.pt'

!ls -lh /content/runs/mobility_yolo11n/weights/

# 검증
model_best = YOLO(best)
metrics = model_best.val(data='/content/datasets/mobility_aids/dataset.yaml')
print(f'mAP50: {metrics.box.map50:.3f}')
print(f'mAP50-95: {metrics.box.map:.3f}')

In [ ]:
# 7. Google Drive에 모델 저장
import shutil, os

save_dir = '/content/drive/MyDrive/sbms-pi/models'
os.makedirs(save_dir, exist_ok=True)

shutil.copy(best, f'{save_dir}/mobility_yolo11n_best.pt')
shutil.copy(last, f'{save_dir}/mobility_yolo11n_last.pt')

print(f'모델 저장 완료 → {save_dir}')
!ls -lh /content/drive/MyDrive/sbms-pi/models/

## 완료 후
1. Drive에서 `sbms-pi/models/mobility_yolo11n_best.pt` 다운로드
2. sola-1으로 전송:
```bash
scp mobility_yolo11n_best.pt admin@192.168.10.100:/home/admin/gunpo/docker/core/
```
3. `.env`에서 모델 경로 설정 후 cv_ffmpeg 서비스 재시작